# StreamAgent Eval
Run cells top to bottom. Results persist to Google Drive so the session can be interrupted and resumed.

In [ ]:
# Cell 1 — Mount Drive and clone repo
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/streamagent'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)

import subprocess

REPO_URL = 'https://github.com/Kaboom2025/StreamingAgenticInference.git'
REPO_DIR = '/content/StreamingAgenticInference'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

import sys
sys.path.insert(0, REPO_DIR)

print('Repo ready at', REPO_DIR)

In [ ]:
# Cell 2 — Install dependencies
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python \
    --upgrade --quiet --no-cache-dir

!pip install -r /content/StreamingAgenticInference/requirements-eval.txt --quiet

import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip() or 'not found — check Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3 — Download model (saved to Drive, skipped on re-runs)
from huggingface_hub import hf_hub_download
import os

MODEL_DRIVE_PATH = f'{DRIVE_DIR}/models/Qwen3-8B-Q4_K_M.gguf'
MODEL_SYMLINK    = '/content/Qwen3-8B-Q4_K_M.gguf'

if not os.path.exists(MODEL_DRIVE_PATH):
    print('Downloading Qwen3-8B Q4_K_M (~5.5GB)...')
    os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
    hf_hub_download(
        repo_id='bartowski/Qwen_Qwen3-8B-GGUF',
        filename='Qwen3-8B-Q4_K_M.gguf',
        local_dir=f'{DRIVE_DIR}/models',
    )
    print('Download complete')
else:
    print('Model already on Drive, skipping download')

if not os.path.exists(MODEL_SYMLINK):
    os.symlink(MODEL_DRIVE_PATH, MODEL_SYMLINK)

print('Model at:', MODEL_SYMLINK)

In [ ]:
# Cell 4 — Download ALFWorld data (saved to Drive, skipped on re-runs)
import os

ALFWORLD_DRIVE = f'{DRIVE_DIR}/alfworld_data'
os.makedirs(ALFWORLD_DRIVE, exist_ok=True)
os.environ['ALFWORLD_DATA'] = ALFWORLD_DRIVE

if not os.listdir(ALFWORLD_DRIVE):
    print('Downloading ALFWorld data (~1.5GB)...')
    !alfworld-download
    print('Done')
else:
    print('ALFWorld data already on Drive, skipping download')

In [ ]:
# Cell 5 — GridWorld eval (primary paper numbers)
# Expected runtime: 60-90 min on T4, 30-45 min on A100
# stream < chunk_1 < chunk_3 < chunk_5 with delta > 5 tokens = hypothesis holds
import subprocess, os

RESULTS_DIR = f'{DRIVE_DIR}/results/gridworld'
os.makedirs(RESULTS_DIR, exist_ok=True)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

subprocess.run([
    'python', '-m', 'streamagent.scripts.run_gridworld_eval',
    '--model-id', '/content/Qwen3-8B-Q4_K_M.gguf',
    '--n-episodes', '20',
], cwd=REPO_DIR, env=env)

In [ ]:
# Cell 6 — ALFWorld eval
# Run only after Cell 5 completes and GridWorld numbers look right.
# --resume skips already-completed tasks if the session dies mid-run.
import subprocess, os

ALFWORLD_RESULTS = f'{DRIVE_DIR}/results/alfworld'
os.makedirs(ALFWORLD_RESULTS, exist_ok=True)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['ALFWORLD_DATA'] = ALFWORLD_DRIVE

subprocess.run([
    'python', '-m', 'streamagent.scripts.run_alfworld_eval',
    '--model-id', '/content/Qwen3-8B-Q4_K_M.gguf',
    '--config', f'{REPO_DIR}/configs/alfworld_eval.yaml',
    '--results-dir', ALFWORLD_RESULTS,
    '--resume',
], cwd=REPO_DIR, env=env)

In [ ]:
# Cell 7 — Print results tables
import json
from pathlib import Path

# GridWorld
gw_results = {}
for f in sorted(Path(RESULTS_DIR).glob('*.json')):
    m = json.loads(f.read_text())
    scenario = m['scenario']
    agent = m['agent']
    gw_results.setdefault(scenario, {})[agent] = m['mean_recovery_tokens']

print('\n=== GRIDWORLD RESULTS ===')
print(f"{'scenario':<20} {'stream':>10} {'chunk_1':>10} {'chunk_3':>10} {'chunk_5':>10}")
print('-' * 60)
for scenario, agents in gw_results.items():
    print(f"{scenario:<20} "
          f"{agents.get('stream', 0):>10.1f} "
          f"{agents.get('chunk_1', 0):>10.1f} "
          f"{agents.get('chunk_3', 0):>10.1f} "
          f"{agents.get('chunk_5', 0):>10.1f}")

# ALFWorld
alf_no_fail   = []
alf_with_fail = []
for f in sorted(Path(ALFWORLD_RESULTS).glob('*.json')):
    m = json.loads(f.read_text())
    (alf_with_fail if m['had_failures'] else alf_no_fail).append(m)

def completion_rate(episodes):
    if not episodes:
        return 0.0
    return sum(e['solved'] for e in episodes) / len(episodes)

print('\n=== ALFWORLD RESULTS ===')
print(f"{'subset':<25} {'n':>6} {'completion_rate':>16}")
print('-' * 50)
print(f"{'no_failures':<25} {len(alf_no_fail):>6} {completion_rate(alf_no_fail):>15.1%}")
print(f"{'with_failures':<25} {len(alf_with_fail):>6} {completion_rate(alf_with_fail):>15.1%}")
all_alf = alf_no_fail + alf_with_fail
print(f"{'overall':<25} {len(all_alf):>6} {completion_rate(all_alf):>15.1%}")